# Arctic Wolf Ticket API — Getting Started Notebook

Python target: **3.14.6**

This notebook is a practical, API-first starter for working with the Arctic Wolf Ticket API. It is written for builders who want copyable Python examples, safe defaults, and a small client wrapper they can extend.

Scope:

- Authenticate with a bearer token.
- List tickets for an organization.
- Filter tickets by status, priority, type, assignee, and timestamps.
- Retrieve a ticket by ID.
- Add a public comment to a ticket.
- Close a ticket with an optional comment.
- Retrieve a pre-signed attachment download URL.
- Handle pagination and API errors.

Out of scope:

- ITSM synchronization design.
- ServiceNow, Jira, Zendesk, or other ticketing platform integration logic.
- Webhook orchestration.
- Production secret management architecture.

## API surface used in this notebook

Base URLs are region-specific. Set the correct host for your organization’s deployment in the environment rather than hard-coding it in this notebook.

Endpoints covered:

| Method | Path | Purpose |
|---|---|---|
| `GET` | `/api/v1/organizations/{organizationUuid}/tickets` | List tickets for an organization. |
| `GET` | `/api/v1/organizations/{organizationUuid}/tickets/{ticketId}` | Retrieve one ticket by ID. |
| `POST` | `/api/v1/organizations/{organizationUuid}/tickets/{ticketId}/comments` | Add a comment to a ticket. |
| `POST` | `/api/v1/organizations/{organizationUuid}/tickets/{ticketId}/close` | Close a ticket. |
| `GET` | `/api/v1/organizations/{organizationUuid}/tickets/{ticketId}/attachments/{attachmentId}` | Get a pre-signed attachment download URL. |

Authentication uses HTTP bearer auth:

```http
Authorization: Bearer <token>
```


## Install dependencies

This notebook only requires `requests`.

## Configuration

Set these values before running API calls.

Recommended local workflow:

1. Store the bearer token in an environment variable.
2. Store the organization UUID in an environment variable.
3. Set the deployment endpoint in the environment.

Required environment variables:

- `AW_TICKET_API_TOKEN` or `PAK_TOKEN`
- `AW_ORGANIZATION_UUID`

Optional environment variables:

- `REGIONAL_ENDPOINT` (host for your deployment)


In [3]:
import json
import os
from pathlib import Path
from getpass import getpass
import dotenv

# Load environment variables from .env
dotenv_path = Path.cwd() / ".env"
print(f"Loading environment from: {dotenv_path}")
dotenv.load_dotenv(dotenv_path, override=True)


def resolve_ticket_base_url(pod: str | None = None, env_url: str | None = None) -> str:
    resolved_pod = (pod or os.getenv("POD") or "us001").strip().lower()
    if env_url:
        return env_url.rstrip("/")

    spec_path = Path.cwd() / "api_definitions" / "ticket_api.json"
    if spec_path.exists():
        with spec_path.open("r", encoding="utf-8") as handle:
            spec = json.load(handle)
        for server in spec.get("servers", []):
            server_url = server.get("url", "")
            description = (server.get("description") or "").lower()
            if resolved_pod in description:
                return server_url.rstrip("/")
            if resolved_pod in server_url.lower():
                return server_url.rstrip("/")

    return f"https://ticket-api.managedgw.{resolved_pod}-prod.arcticwolf.net"


BASE_URL = resolve_ticket_base_url(
    pod=os.getenv("POD"),
    env_url=(os.getenv("REGIONAL_ENDPOINT") or os.getenv("BASE_URL") or "").rstrip("/"),
)

TOKEN = os.getenv("AW_TICKET_API_TOKEN") or os.getenv("PAK_TOKEN")
if not TOKEN:
    TOKEN = getpass("PAK_TOKEN: ")

ORGANIZATION_UUID = os.getenv("AW_ORGANIZATION_UUID")
if not ORGANIZATION_UUID:
    ORGANIZATION_UUID = input("AW_ORGANIZATION_UUID: ").strip()

print("✅ Configuration loaded")
print(f"Base URL: {BASE_URL}")
print(f"Organization UUID: {ORGANIZATION_UUID}")


Loading environment from: /workspaces/Data_Service_API/.env
✅ Configuration loaded
Base URL: https://ticket-api.managedgw.us001-prod.arcticwolf.net
Organization UUID: cbcfa21a-42e5-4087-849d-7a97cbcc10a5


## Core client

The client below keeps the examples short while preserving explicit request behavior.

Design choices:

- Uses a persistent `requests.Session`.
- Applies bearer token auth once at session creation.
- Converts list query parameters into comma-separated strings because the API accepts single values or comma-separated lists for filters such as `status`, `priority`, and `type`.
- Raises a custom exception containing HTTP status, API error code, API error description, and raw response text when available.

In [4]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Any, Iterable, Optional
from urllib.parse import urlparse
import requests
import time


@dataclass
class TicketApiError(Exception):
    status_code: int
    code: str | None = None
    description: str | None = None
    response_text: str | None = None

    def __str__(self) -> str:
        parts = [f"HTTP {self.status_code}"]
        if self.code:
            parts.append(f"code={self.code}")
        if self.description:
            parts.append(f"description={self.description}")
        if self.response_text and not self.description:
            parts.append(f"response={self.response_text[:500]}")
        return " | ".join(parts)


class ArcticWolfTicketApiClient:
    """Arctic Wolf Ticket API client with retry logic and URL validation."""

    def __init__(self, base_url: str, token: str, timeout_seconds: int = 30, max_retries: int = 3) -> None:
        self.base_url = base_url.rstrip("/")
        self.timeout_seconds = timeout_seconds
        self.max_retries = max_retries
        self.session = requests.Session()
        self.session.headers.update({
            "Authorization": f"Bearer {token}",
            "Accept": "application/json",
            "Content-Type": "application/json",
            "User-Agent": "aw-ticket-api-getting-started-notebook/1.0",
        })

    @staticmethod
    def _normalize_params(params: dict[str, Any] | None) -> dict[str, Any]:
        """Normalize query parameters."""
        if not params:
            return {}
        normalized: dict[str, Any] = {}
        for key, value in params.items():
            if value is None:
                continue
            if isinstance(value, bool):
                normalized[key] = str(value).lower()
            elif isinstance(value, (list, tuple, set)):
                normalized[key] = ",".join(str(item) for item in value)
            else:
                normalized[key] = value
        return normalized

    def _request(
        self,
        method: str,
        path: str,
        *,
        params: dict[str, Any] | None = None,
        json_body: dict[str, Any] | None = None,
    ) -> Any:
        """Make HTTP request with retry logic for transient errors."""
        for attempt in range(self.max_retries):
            try:
                url = f"{self.base_url}{path}"
                response = self.session.request(
                    method=method,
                    url=url,
                    params=self._normalize_params(params),
                    json=json_body,
                    timeout=self.timeout_seconds,
                )

                if response.status_code >= 400:
                    code = None
                    description = None
                    try:
                        body = response.json()
                        code = body.get("code")
                        description = body.get("description")
                    except ValueError:
                        body = None
                    raise TicketApiError(
                        status_code=response.status_code,
                        code=code,
                        description=description,
                        response_text=response.text,
                    )

                if response.status_code == 204 or not response.content:
                    return None
                return response.json()
            except TicketApiError as e:
                # Retry on transient errors (429, 5xx)
                if e.status_code in (429, 500, 502, 503, 504) and attempt < self.max_retries - 1:
                    wait_time = 2 ** attempt
                    print(f"⚠️ {e.status_code} error. Retrying in {wait_time}s... (attempt {attempt + 1}/{self.max_retries})")
                    time.sleep(wait_time)
                else:
                    raise
            except requests.exceptions.RequestException as e:
                if attempt < self.max_retries - 1:
                    wait_time = 2 ** attempt
                    print(f"⚠️ Connection error. Retrying in {wait_time}s... (attempt {attempt + 1}/{self.max_retries})")
                    time.sleep(wait_time)
                else:
                    raise
        return None

    def list_tickets(
        self,
        organization_uuid: str,
        *,
        status: str | Iterable[str] | None = None,
        assignee_by_email: str | Iterable[str] | None = None,
        assignee_by_first_name: str | Iterable[str] | None = None,
        assignee_by_last_name: str | Iterable[str] | None = None,
        updated_before: str | None = None,
        updated_after: str | None = None,
        created_before: str | None = None,
        created_after: str | None = None,
        priority: str | Iterable[str] | None = None,
        ticket_type: str | Iterable[str] | None = None,
        offset: int = 0,
        limit: int = 20,
        include_comments: bool = False,
    ) -> dict[str, Any]:
        path = f"/api/v1/organizations/{organization_uuid}/tickets"
        params = {
            "status": status,
            "assigneeByEmail": assignee_by_email,
            "assigneeByFirstName": assignee_by_first_name,
            "assigneeByLastName": assignee_by_last_name,
            "updatedBefore": updated_before,
            "updatedAfter": updated_after,
            "createdBefore": created_before,
            "createdAfter": created_after,
            "priority": priority,
            "type": ticket_type,
            "offset": offset,
            "limit": limit,
            "includeComments": include_comments,
        }
        return self._request("GET", path, params=params)

    def get_ticket(
        self,
        organization_uuid: str,
        ticket_id: int,
        *,
        include_comments: bool = False,
    ) -> dict[str, Any]:
        path = f"/api/v1/organizations/{organization_uuid}/tickets/{ticket_id}"
        return self._request("GET", path, params={"includeComments": include_comments})

    def add_comment(
        self,
        organization_uuid: str,
        ticket_id: int,
        body: str,
    ) -> dict[str, Any]:
        path = f"/api/v1/organizations/{organization_uuid}/tickets/{ticket_id}/comments"
        return self._request("POST", path, json_body={"body": body})

    def close_ticket(
        self,
        organization_uuid: str,
        ticket_id: int,
        comment: str | None = None,
    ) -> dict[str, Any]:
        path = f"/api/v1/organizations/{organization_uuid}/tickets/{ticket_id}/close"
        json_body = {"comment": comment} if comment else {}
        return self._request("POST", path, json_body=json_body)

    def get_attachment_url(
        self,
        organization_uuid: str,
        ticket_id: int,
        attachment_id: int,
    ) -> dict[str, Any]:
        path = f"/api/v1/organizations/{organization_uuid}/tickets/{ticket_id}/attachments/{attachment_id}"
        return self._request("GET", path)


## Create the API client

In [5]:
client = ArcticWolfTicketApiClient(BASE_URL, TOKEN)

## Example 1 — List tickets

This retrieves the first page of tickets for the organization.

Pagination defaults:

- `offset`: `0`
- `limit`: `20`

The API limit parameter supports values from `1` to `100`.

In [6]:
try:
    page = client.list_tickets(
        ORGANIZATION_UUID,
        offset=0,
        limit=20,
        include_comments=False,
    )
    print(f"Returned: {len(page.get('results', []))}")
    print(f"Meta: {page.get('meta')}")
    page.get("results", [])[:3]
except TicketApiError as exc:
    print(exc)

Returned: 20
Meta: {'offset': 0, 'limit': 20, 'total': 1680}


## Example 2 — List open customer-action tickets

Ticket status mapping:

- `OPEN`, `NEW`, `HOLD`: With Arctic Wolf
- `PENDING`: With Customer
- `CLOSED`: Closed
- `OPEN`, `NEW`, `HOLD`, `PENDING`: Open

This example returns tickets that are pending customer action.

In [7]:
try:
    pending_customer_tickets = client.list_tickets(
        ORGANIZATION_UUID,
        status="PENDING",
        limit=20,
        include_comments=False,
    )
    results = pending_customer_tickets.get("results", [])
    print(f"✅ Found {len(results)} pending customer tickets")
    results
except TicketApiError as exc:
    print(exc)

✅ Found 20 pending customer tickets


## Example 3 — Filter by priority, type, and creation time

Supported priorities:

- `LOW`
- `NORMAL`
- `HIGH`
- `URGENT`

Supported ticket types:

- `QUESTION`
- `INCIDENT`
- `PROBLEM`
- `TASK`

Timestamp filters use ISO 8601 UTC values such as `2026-03-01T00:00:00Z`.

In [8]:
try:
    high_priority_incidents = client.list_tickets(
        ORGANIZATION_UUID,
        priority=["HIGH", "URGENT"],
        ticket_type="INCIDENT",
        created_after="2026-01-01T00:00:00Z",
        limit=50,
        include_comments=False,
    )
    high_priority_incidents.get("results", [])
    print(f"✅ Found {len(results)} incident customer tickets marked as HIGH or URGENT priority")
except TicketApiError as exc:
    print(exc)

✅ Found 20 incident customer tickets marked as HIGH or URGENT priority


## Example 4 — Fetch all pages

Use this helper when you want all matching tickets instead of one page.

Guardrails:

- Uses API pagination via `offset` and `limit`.
- Sets `limit=100`, the documented maximum for list requests.
- Stops when the API returns fewer records than requested or when `offset + returned >= total`.

In [9]:
def fetch_all_tickets(
    client: ArcticWolfTicketApiClient,
    organization_uuid: str,
    *,
    limit: int = 100,
    max_pages: int = 100,
    **filters: Any,
) -> list[dict[str, Any]]:
    all_results: list[dict[str, Any]] = []
    offset = 0

    for _ in range(max_pages):
        page = client.list_tickets(
            organization_uuid,
            offset=offset,
            limit=limit,
            **filters,
        )
        results = page.get("results", [])
        meta = page.get("meta", {})
        all_results.extend(results)

        total = meta.get("total")
        returned = len(results)
        offset += returned

        if returned < limit:
            break
        if isinstance(total, int) and offset >= total:
            break

    return all_results


try:
    open_tickets = fetch_all_tickets(
        client,
        ORGANIZATION_UUID,
        status=["OPEN", "NEW", "HOLD", "PENDING"],
        include_comments=False,
    )
    print(f"Fetched {len(open_tickets)} open tickets")
except TicketApiError as exc:
    print(exc)

Fetched 34 open tickets


## Example 5 — Convert ticket results into a DataFrame

This is useful for quick inspection, export, or sorting. It is not required for API usage.

In [10]:
import pandas as pd
import json


def tickets_to_dataframe(tickets: list[dict[str, Any]]) -> pd.DataFrame:
    rows = []
    for ticket in tickets:
        assignee = ticket.get("assignee") or {}
        attachments = ticket.get("attachments") or []
        attachment_ids = [att.get("id") for att in attachments if not att.get("deleted")]
        rows.append({
            "id": ticket.get("id"),
            "title": ticket.get("title"),
            "status": ticket.get("status"),
            "priority": ticket.get("priority"),
            "type": ticket.get("type"),
            "createdAt": ticket.get("createdAt"),
            "updatedAt": ticket.get("updatedAt"),
            "commentCount": ticket.get("commentCount"),
            "attachmentCount": ticket.get("attachmentCount"),
            "attachmentIds": attachment_ids if attachment_ids else None,
            "assigneeEmail": assignee.get("email"),
            "assigneeName": " ".join(
                part for part in [assignee.get("firstName"), assignee.get("lastName")] if part
            ) or None,
        })
    return pd.DataFrame(rows)


try:
    sample_page = client.list_tickets(ORGANIZATION_UUID, limit=20)
    results = sample_page.get("results", [])
    print(f"✅ Generated DataFrame with {len(results)} tickets\n")
    
    # Display DataFrame
    df = tickets_to_dataframe(results)
    display(df)
except TicketApiError as exc:
    print(exc)

✅ Generated DataFrame with 20 tickets



,id,title,status,priority,type,createdAt,updatedAt,commentCount,attachmentCount,attachmentIds,assigneeEmail,assigneeName
0,17763977,[MEDIUM]Incident: Anomalous sign-in from a low...,PENDING,NORMAL,INCIDENT,2026-07-20T18:58:45Z,2026-07-20T18:58:47Z,1,1.0,None,kyle.hatlestad@arcticwolf.net,Kyle Hatlestad
1,17755066,[HIGH]Incident: Filename patterns associated w...,PENDING,HIGH,INCIDENT,2026-07-18T22:11:25Z,2026-07-18T22:11:37Z,1,NaN,None,kyle.hatlestad@arcticwolf.net,Kyle Hatlestad
2,17755035,[MEDIUM] Incident: Office 365: Evasive inbox r...,PENDING,NORMAL,INCIDENT,2026-07-18T21:45:41Z,2026-07-18T21:46:00Z,1,1.0,None,kyle.hatlestad@arcticwolf.net,Kyle Hatlestad
3,17755016,[MEDIUM]Incident: Member added to critical AD ...,PENDING,NORMAL,INCIDENT,2026-07-18T21:27:47Z,2026-07-18T21:27:51Z,1,1.0,None,kyle.hatlestad@arcticwolf.net,Kyle Hatlestad
4,17754907,[MEDIUM]Incident: Aurora Defense: PowerShell E...,PENDING,NORMAL,INCIDENT,2026-07-18T20:05:29Z,2026-07-18T20:05:38Z,1,1.0,None,kyle.hatlestad@arcticwolf.net,Kyle Hatlestad
5,17754796,[MEDIUM]Incident: Abnormal behavior: unusual a...,PENDING,NORMAL,INCIDENT,2026-07-18T18:23:30Z,2026-07-18T18:23:41Z,1,1.0,None,kyle.hatlestad@arcticwolf.net,Kyle Hatlestad
6,17754791,[MEDIUM]Incident: Mimikatz Lsadump Invocation ...,PENDING,NORMAL,INCIDENT,2026-07-18T18:11:49Z,2026-07-18T18:11:55Z,1,1.0,None,kyle.hatlestad@arcticwolf.net,Kyle Hatlestad
7,17754789,[MEDIUM]Incident: Mimikatz Initialization Dete...,PENDING,NORMAL,INCIDENT,2026-07-18T18:10:12Z,2026-07-18T18:10:24Z,1,1.0,None,kyle.hatlestad@arcticwolf.net,Kyle Hatlestad
8,17754769,[MEDIUM]Incident: File Create by WinRAR in Unc...,PENDING,NORMAL,INCIDENT,2026-07-18T17:55:26Z,2026-07-18T17:55:32Z,1,1.0,None,kyle.hatlestad@arcticwolf.net,Kyle Hatlestad
9,17754740,[HIGH]Incident: Backdoor.Win32.Qakbot.E (Initi...,PENDING,HIGH,INCIDENT,2026-07-18T17:39:33Z,2026-07-18T17:39:39Z,1,1.0,None,kyle.hatlestad@arcticwolf.net,Kyle Hatlestad


## Example 6 — Retrieve one ticket by ID

Set `TICKET_ID` to a real ticket ID from a previous list response.

Set `include_comments=True` when you need comments and attachment metadata in the response.

In [11]:
TICKET_ID = 17614272  # Replace with a real ticket ID.

try:
    ticket = client.get_ticket(
        ORGANIZATION_UUID,
        TICKET_ID,
        include_comments=True,
    )
    
    # Build markdown output
    markdown_output = f"""# Ticket #{ticket.get('id')} — {ticket.get('title')}

## Overview

| Field | Value |
|-------|-------|
| Status | {ticket.get('status')} |
| Priority | {ticket.get('priority')} |
| Type | {ticket.get('type')} |
| Created | {ticket.get('createdAt')} |
| Updated | {ticket.get('updatedAt')} |

## Details

**Organization UUID:** {ticket.get('organizationUuid')}

**Description:** {ticket.get('description', 'N/A')}

**Comments:** {ticket.get('commentCount', 0)}

**Attachments:** {ticket.get('attachmentCount', 0)}
"""
    
    # Add assignee if available
    assignee = ticket.get('assignee')
    if assignee:
        markdown_output += f"""
## Assignee

- **Name:** {assignee.get('firstName', '')} {assignee.get('lastName', '')}
- **Email:** {assignee.get('email')}
"""
    
    # Add comments if available
    comments = ticket.get('comments', [])
    if comments:
        markdown_output += f"""
## Comments ({len(comments)})

"""
        for comment in comments:
            author = comment.get('author', {})
            markdown_output += f"""### {author.get('firstName', '')} {author.get('lastName', '')} — {comment.get('createdAt')}

**Type:** {comment.get('type', 'N/A')}

{comment.get('body', '')}

---

"""
    
    # Add attachments if available
    attachments = ticket.get('attachments', [])
    if attachments:
        markdown_output += f"""
## Attachments ({len(attachments)})

| ID | Filename | Type | Created | Status |
|----|-----------|----|---------|--------|
"""
        for att in attachments:
            status = "🗑️ Deleted" if att.get('deleted') else "✅ Active"
            markdown_output += f"| {att.get('id')} | {att.get('filename', 'N/A')} | {att.get('contentType', 'N/A')} | {att.get('createdAt', 'N/A')} | {status} |\n"
    
    # Add full JSON
    markdown_output += f"""
## Full Ticket Object (JSON)

```json
{json.dumps(ticket, indent=2)}
```
"""
    
    print(markdown_output)
    
except TicketApiError as exc:
    print(exc)

# Ticket #17614272 — [HIGH]Incident: Filename patterns associated with SharpHound/BloodHound - Sample Co Llp

## Overview

| Field | Value |
|-------|-------|
| Status | CLOSED |
| Priority | HIGH |
| Type | INCIDENT |
| Created | 2026-06-27T22:11:53Z |
| Updated | 2026-07-07T23:02:39Z |

## Details

**Organization UUID:** None

**Description:** ## Summary

- BloodHound reconnaissance tool detected creating Active Directory enumeration files on desktop1 (10.171.170.101)
- Immediately isolate the affected host and investigate for unauthorized access

### What is it?

Arctic Wolf observed BloodHound/SharpHound execution on desktop1 at 2026-06-27T22:01:31Z UTC. The DecryptedSharpHound.exe process created multiple JSON files containing Active Directory reconnaissance data including groups, users, computers, and domain information.

This activity is considered malicious because BloodHound is a reconnaissance tool commonly used by threat actors to map Active Directory environments and identi

## Example 7 — Add a comment to a ticket

The request body requires `body`.

The comment body supports up to 65,535 characters.

In [12]:
TICKET_ID = 12345  # Replace with a real ticket ID.
COMMENT_BODY = "API test comment from getting started notebook."

DRY_RUN = True

if DRY_RUN:
    print("DRY_RUN=True. No comment was added.")
    print({"ticket_id": TICKET_ID, "body": COMMENT_BODY})
else:
    try:
        added_comment = client.add_comment(
            ORGANIZATION_UUID,
            TICKET_ID,
            COMMENT_BODY,
        )
        added_comment
    except TicketApiError as exc:
        print(exc)

DRY_RUN=True. No comment was added.
{'ticket_id': 12345, 'body': 'API test comment from getting started notebook.'}


## Example 8 — Close a ticket

Closing a ticket uses a `POST` request and accepts an optional `comment` field.

Keep `DRY_RUN=True` until you intentionally want to close a real ticket.

In [13]:
TICKET_ID = 12345  # Replace with a real ticket ID.
CLOSE_COMMENT = "Closing via Ticket API after validation."

DRY_RUN = True

if DRY_RUN:
    print("DRY_RUN=True. No ticket was closed.")
    print({"ticket_id": TICKET_ID, "comment": CLOSE_COMMENT})
else:
    try:
        closed_ticket = client.close_ticket(
            ORGANIZATION_UUID,
            TICKET_ID,
            comment=CLOSE_COMMENT,
        )
        closed_ticket
    except TicketApiError as exc:
        print(exc)

DRY_RUN=True. No ticket was closed.
{'ticket_id': 12345, 'comment': 'Closing via Ticket API after validation.'}


## Example 9 — Retrieve an attachment download URL

The attachment endpoint returns a pre-signed URL. The URL expires after a limited time.

Use attachment metadata from a ticket response where `includeComments=True`.

Do not request download URLs for attachments where `deleted=True`.

In [14]:
TICKET_ID = 17614272       # Replace with a real ticket ID.
ATTACHMENT_ID = 52265706791451   # Replace with a real attachment ID.

try:
    attachment_url_response = client.get_attachment_url(
        ORGANIZATION_UUID,
        TICKET_ID,
        ATTACHMENT_ID,
    )
    attachment_url_response
except TicketApiError as exc:
    print(exc)

## Example 10 — Download an attachment from the pre-signed URL

This separate request does not use the Ticket API bearer token. It uses the returned pre-signed URL.

Only run this when you trust the file type and destination path.

In [15]:
from pathlib import Path
from urllib.parse import urlparse


def download_presigned_url(url: str, destination_path: str | Path, allowed_domains: list[str] | None = None) -> Path:
    """Download from presigned URL with domain validation.
    
    Args:
        url: Presigned download URL
        destination_path: Local file path to save to
        allowed_domains: List of allowed domain names (e.g., ['*.arcticwolf.com', 'storage.api.com'])
                         If None, no domain validation is performed (not recommended)
    
    Returns:
        Path to downloaded file
    
    Raises:
        ValueError: If URL domain is not in allowed list
    """
    # Validate URL origin if allowed_domains is specified
    if allowed_domains:
        parsed = urlparse(url)
        hostname = parsed.hostname or ""
        is_allowed = any(
            domain.startswith("*.") and hostname.endswith(domain[1:]) or
            hostname == domain
            for domain in allowed_domains
        )
        if not is_allowed:
            raise ValueError(
                f"URL domain '{hostname}' not in allowed domains: {allowed_domains}"
            )
    else:
        print("⚠️ WARNING: Downloading from URL without domain validation. This is a security risk.")

    destination = Path(destination_path)
    with requests.get(url, stream=True, timeout=60) as response:
        response.raise_for_status()
        with destination.open("wb") as file_obj:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    file_obj.write(chunk)
    return destination


# Example usage with domain validation:
# response = client.get_attachment_url(ORGANIZATION_UUID, TICKET_ID, ATTACHMENT_ID)
# saved_path = download_presigned_url(
#     response["url"],
#     "attachment.bin",
#     allowed_domains=["*.arcticwolf.com", "s3.amazonaws.com"]  # Whitelist trusted domains
# )


## Common error responses

The API returns a JSON error envelope with at least `code`, and often `description`.

Common HTTP responses:

| Status | Meaning |
|---:|---|
| `400` | Invalid request. Example: `limit` greater than the allowed maximum. |
| `401` | Missing, invalid, or expired authentication token. |
| `403` | Authenticated but insufficient permissions. |
| `404` | Ticket or attachment was not found. |
| `500` | Internal server error. |

The `TicketApiError` class above extracts `code` and `description` when present.

## Minimal copy-paste script

This is a compact version for vibe-coded prototypes.

In [17]:
import json
import os
import requests
from pathlib import Path

# Reuse the same environment-driven base URL resolution as the earlier setup cell

def resolve_ticket_base_url(pod: str | None = None, env_url: str | None = None) -> str:
    resolved_pod = (pod or os.getenv("POD") or "us001").strip().lower()
    if env_url:
        return env_url.rstrip("/")

    spec_path = Path.cwd() / "api_definitions" / "ticket_api.json"
    if spec_path.exists():
        with spec_path.open("r", encoding="utf-8") as handle:
            spec = json.load(handle)
        for server in spec.get("servers", []):
            server_url = server.get("url", "")
            description = (server.get("description") or "").lower()
            if resolved_pod in description:
                return server_url.rstrip("/")
            if resolved_pod in server_url.lower():
                return server_url.rstrip("/")

    return f"https://ticket-api.managedgw.{resolved_pod}-prod.arcticwolf.net"


BASE_URL = resolve_ticket_base_url(
    pod=os.getenv("POD"),
    env_url=(os.getenv("REGIONAL_ENDPOINT") or os.getenv("BASE_URL") or "").rstrip("/"),
)
TOKEN = os.getenv("AW_TICKET_API_TOKEN") or os.getenv("PAK_TOKEN")
ORGANIZATION_UUID = os.getenv("AW_ORGANIZATION_UUID")

if not TOKEN or not ORGANIZATION_UUID:
    raise ValueError("Set AW_TICKET_API_TOKEN/PAK_TOKEN and AW_ORGANIZATION_UUID before running this example.")

headers = {
    "Authorization": f"Bearer {TOKEN}",
    "Accept": "application/json",
    "Content-Type": "application/json",
}

response = requests.get(
    f"{BASE_URL}/api/v1/organizations/{ORGANIZATION_UUID}/tickets",
    headers=headers,
    params={
        "status": "OPEN,PENDING",
        "priority": "HIGH,URGENT",
        "limit": 20,
        "offset": 0,
        "includeComments": "false",
    },
    timeout=30,
)
response.raise_for_status()

tickets = response.json()
tickets


{'results': [{'id': 17755066,
   'title': '[HIGH]Incident: Filename patterns associated with SharpHound/BloodHound - Sample Co Llp',
   'description': '## Summary\n\n- BloodHound reconnaissance tool detected creating Active Directory enumeration files on desktop1 (10.171.170.101)\n- Immediately disable the NT AUTHORITY\\\\SYSTEM account activity and investigate the source of the DecryptedSharpHound.exe execution\n\n### What is it?\n\nArctic Wolf observed BloodHound/SharpHound reconnaissance activity at 2026-07-18T22:01:10Z UTC. The DecryptedSharpHound.exe process created multiple JSON files containing Active Directory enumeration data, including user accounts, computers, groups, and domain trust information.\n\nThis activity is considered malicious because BloodHound is a reconnaissance tool commonly used by threat actors to map Active Directory environments and identify privilege escalation paths. The tool systematically enumerates domain objects to create attack graphs that reveal po

## Safe extension points

Useful next steps for prototype code:

- Add retries for transient `429`, `500`, `502`, `503`, and `504` responses if those are observed in your environment.
- Add structured logging around method, path, status code, and elapsed time. Do not log tokens.
- Store tokens in a secret manager for deployed workflows.
- Add idempotency checks before commenting or closing tickets.
- Keep destructive operations behind an explicit `DRY_RUN` flag until tested.